# M2.5 · Random forest ve gradient boosting

Zayıf modellerden güçlü model. Gradient boosting'in iki turunu ve bagging'in varyans azaltmasını elle hesaplıyor; saf Python'da tek özellikli boosting yazıyor; NumPy'da regresyon stump'ını cumsum ile döngüsüz buluyor; PyTorch'ta pseudo-residual'ları autograd'a hesaplatarak keyfi kayıpla boosting yapıyor ve en sonda B bootstrap ağacını tek tensörde eğitiyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m2-5-ensembles.html](https://m-gonul.github.io/pytorch-lab/m2-5-ensembles.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m2_5_ensembles.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch/sklearn kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\", \"sklearn\"}))\n\n    def st():\n        t, l, r = need(ns, \"fit_stump\")([1, 2, 3, 4], [-1, -1, 1, 1])\n        close([t, l, r], [2.5, -1, 1])\n        t2, l2, r2 = need(ns, \"fit_stump\")([3, 3, 3], [1.0, 2.0, 3.0])\n        ensure(t2 is None, \"tek farklı x → None\"); close([l2, r2], [2, 2])\n    R.test(\"fit_stump: elle örnek, bölünemez durum\", st)\n\n    def hand():\n        m = need(ns, \"gbm_fit\")([1, 2, 3, 4], [1, 1, 3, 3], 2, 0.5)\n        close([need(ns, \"gbm_predict\")(m, x) for x in [1, 2, 3, 4]], [1.25, 1.25, 2.75, 2.75])\n    R.test(\"2 tur: [1.25, 1.25, 2.75, 2.75]\", hand)\n\n    def curve():\n        rng = random.Random(0)\n        xs = [rng.uniform(0, 6) for _ in range(80)]\n        ys = [math.sin(x) + rng.gauss(0, 0.1) for x in xs]\n        mses = []\n        for rounds in (1, 10, 100):\n            m = need(ns, \"gbm_fit\")(xs, ys, rounds, 0.3)\n            mses.append(sum((need(ns, \"gbm_predict\")(m, x) - y) ** 2 for x, y in zip(xs, ys)) / 80)\n        ensure(mses[0] > mses[1] > mses[2] and mses[2] < 0.02, f\"eğitim MSE'leri {mses}\")\n        return \"eğitim MSE (1, 10, 100 tur): \" + \", \".join(f\"{v:.4f}\" for v in mses)\n    R.test(\"sin(x)'e uyum: MSE turlarla azalıyor\", curve)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, no_imports, need, ensure\n\n\ndef brute(X, r):\n    n, d = X.shape\n    best = None\n    for j in range(d):\n        v = np.unique(X[:, j])\n        for t in (v[:-1] + v[1:]) / 2:\n            m = X[:, j] <= t\n            g = r[m].sum() ** 2 / m.sum() + r[~m].sum() ** 2 / (~m).sum()\n            if best is None or g > best[0] + 1e-12:\n                best = (g, j, t, r[m].mean(), r[~m].mean())\n    return best[1:]\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"best_reg_stump: döngü yok\", lambda: max_loops(src, 0, func=\"best_reg_stump\"))\n    R.test(\"gbm_np / gbm_predict_np: yalnız tur döngüsü\", lambda: (max_loops(src, 1, func=\"gbm_np\"), max_loops(src, 1, func=\"gbm_predict_np\")))\n    R.test(\"torch/sklearn kullanılmıyor\", lambda: no_imports(src, {\"torch\", \"sklearn\"}))\n    rng = np.random.default_rng(0)\n\n    def st():\n        X = rng.normal(size=(50, 4)).round(2); r = rng.normal(size=50)\n        got, want = need(ns, \"best_reg_stump\")(X, r), brute(X, r)\n        ensure(got[0] == want[0], f\"özellik {got[0]}, {want[0]} bekleniyordu\"); close(got[1:], want[1:])\n        close(need(ns, \"best_reg_stump\")(np.array([[1.], [2], [3], [4]]), np.array([-1., -1, 1, 1]))[1:], [2.5, -1, 1], what=\"elle örnek\")\n    R.test(\"best_reg_stump kaba kuvvetle aynı\", st)\n\n    def fit():\n        X = rng.uniform(-2, 2, size=(600, 3)); y = np.sin(2 * X[:, 0]) + X[:, 1] ** 2 + 0.05 * rng.normal(size=600)\n        Xt = rng.uniform(-2, 2, size=(300, 3)); yt = np.sin(2 * Xt[:, 0]) + Xt[:, 1] ** 2\n        m = need(ns, \"gbm_np\")(X, y, 200, 0.2)\n        p = need(ns, \"gbm_predict_np\")(m, Xt)\n        r2 = 1 - np.mean((p - yt) ** 2) / np.var(yt)\n        ensure(r2 >= 0.9, f\"test R² = {r2:.3f}\")\n        return f\"test R² = {r2:.3f} (stump'ların toplamı: toplamsal model)\"\n    R.test(\"doğrusal olmayan fonksiyonda test R² ≥ 0.9\", fit)\n",
 "d": "import torch\nimport torch.nn.functional as F\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    g = torch.Generator().manual_seed(0)\n    Fv = torch.randn(20, generator=g, dtype=torch.float64)\n    y = (torch.rand(20, generator=g) < 0.5).double()\n\n    def pr():\n        f = need(ns, \"pseudo_residuals\")\n        close(f(Fv, y, lambda a, b: (a - b) ** 2), -2 * (Fv - y), what=\"MSE: −2(F − y)\")\n        close(f(Fv, y, lambda a, b: F.binary_cross_entropy_with_logits(a, b, reduction=\"none\")), y - torch.sigmoid(Fv), what=\"logistic: y − σ(F)\")\n        close(f(Fv, y, lambda a, b: (a - b).abs()), -torch.sign(Fv - y), what=\"L1: −sign(F − y)\")\n    R.test(\"pseudo_residuals: MSE, logistic, L1\", pr)\n\n    def circ():\n        n = 300\n        r = torch.cat([torch.rand(n, generator=g) * 1.0, 1.6 + torch.rand(n, generator=g) * 1.0])\n        t = torch.rand(2 * n, generator=g) * 6.2832\n        X = torch.stack([r * t.cos(), r * t.sin()], 1).double()\n        yy = torch.cat([torch.ones(n), torch.zeros(n)]).double()\n        m = need(ns, \"gbm_classify\")(X, yy, rounds=200, lr=0.3)\n        acc = ((need(ns, \"gbm_logits\")(m, X) > 0).double() == yy).double().mean().item()\n        ensure(acc >= 0.9, f\"doğruluk {acc:.2f}\")\n        return f\"iç içe çemberler, 200 stump: doğruluk {acc:.3f}\"\n    R.test(\"gbm_classify çemberleri ayırıyor\", circ)\n",
 "e": "import torch\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef weighted_stump(X, y, w):\n    n, d = X.shape\n    best = None\n    for j in range(d):\n        v = torch.unique(X[:, j])\n        for t in ((v[:-1] + v[1:]) / 2).tolist():\n            m = X[:, j] <= t\n            WL, WR = w[m].sum(), w[~m].sum()\n            if WL == 0 or WR == 0:\n                continue\n            SL, SR = (w * y)[m].sum(), (w * y)[~m].sum()\n            gn = float(SL ** 2 / WL + SR ** 2 / WR)\n            if best is None or gn > best[0] + 1e-9:\n                best = (gn, j, t, float(SL / WL), float(SR / WR))\n    return best[1:]\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    X = torch.rand(40, 3, generator=g, dtype=torch.float64)\n    y = torch.sin(6 * X[:, 0]) + X[:, 2] + 0.1 * torch.randn(40, generator=g, dtype=torch.float64)\n\n    def eq():\n        gen = torch.Generator().manual_seed(5)\n        j, t, l, r = need(ns, \"bagged_stumps\")(X, y, 12, gen)\n        ensure(tuple(j.shape) == (12,), f\"j shape {tuple(j.shape)}\")\n        idx = torch.randint(40, (12, 40), generator=torch.Generator().manual_seed(5))\n        for b in range(12):\n            w = torch.bincount(idx[b], minlength=40).double()\n            jj, tt, ll, rr = weighted_stump(X, y, w)\n            ensure(int(j[b]) == jj, f\"b={b}: özellik {int(j[b])}, {jj} bekleniyordu\")\n            close([float(t[b]), float(l[b]), float(r[b])], [tt, ll, rr], what=f\"b={b}\")\n        return \"12 bootstrap stump'ın her biri tek tek oturtulanla aynı (aynı generator dizisi)\"\n    R.test(\"her bootstrap stump referansla aynı\", eq)\n\n    def pred():\n        m = need(ns, \"bagged_stumps\")(X, y, 200, torch.Generator().manual_seed(1))\n        p = need(ns, \"bagged_predict\")(m, X)\n        ensure(tuple(p.shape) == (40,), f\"shape {tuple(p.shape)}\")\n        j, t, l, r = m\n        manual = torch.stack([torch.where(X[:, int(j[b])] <= t[b], l[b], r[b]) for b in range(200)]).mean(0)\n        close(p, manual)\n    R.test(\"bagged_predict = stump tahminlerinin ortalaması\", pred)\n    if \"bagged_stumps\" not in ns:\n        return\n    Xb = torch.rand(300, 8, generator=g); yb = torch.sin(6 * Xb[:, 0]) + Xb[:, 1]\n\n    def loop():\n        idx = torch.randint(300, (50, 300), generator=torch.Generator().manual_seed(0))\n        for b in range(50):\n            w = torch.bincount(idx[b], minlength=300).float()\n            Xs, order = torch.sort(Xb, 0); Ws = w[order]; Ss = (w * yb)[order]\n            WL, SL = Ws.cumsum(0)[:-1], Ss.cumsum(0)[:-1]\n            (SL ** 2 / WL.clamp_min(1e-9) + (Ss.sum(0) - SL) ** 2 / (Ws.sum(0) - WL).clamp_min(1e-9)).argmax()\n    t_loop = bench(loop, repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"bagged_stumps\"](Xb, yb, 50, torch.Generator().manual_seed(0)), repeat=3)\n    R.table(\"50 bootstrap stump (n = 300, d = 8)\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for b: vektörize stump (B sıralama)\", fmt_time(t_loop), \"1×\"], [\"senin bagged_stumps (1 sıralama)\", fmt_time(t_vec), f\"{t_loop / t_vec:,.1f}× hızlı\"]])\n"
}

In [ ]:
import importlib, labcheck, checks_m2_5_ensembles as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Tek bir ağaç yüksek varyanslıdır: veriyi biraz değiştir, bambaşka bir ağaç çıkar. İki zıt strateji:

**Bagging / random forest** varyansı düşürür. $B$ bootstrap örneklemi üzerinde $B$ derin ağaç eğitip tahminlerini ortalar. Her birinin varyansı $\sigma^2$, aralarındaki korelasyon $\rho$ ise

$$
\operatorname{Var}\Big[\tfrac1B\textstyle\sum_b f_b\Big]=\rho\sigma^2+\frac{1-\rho}{B}\sigma^2 .
$$

**Boosting** bias'ı düşürür. Sığ ağaçları **sırayla** ekler, her biri bir öncekilerin hatasını düzeltir. Gradient boosting bunu fonksiyon uzayında gradient descent olarak yazar:

$$
F_0=\arg\min_c\sum_i L(y_i,c),\qquad
r_i^{(m)}=-\frac{\partial L(y_i,F)}{\partial F}\Big|_{F=F_{m-1}(x_i)},\qquad
F_m=F_{m-1}+\eta\,h_m,
$$

$h_m$, pseudo-residual'lara oturtulan bir regresyon ağacıdır. MSE'de $r=y-F$ (gerçek residual), logistic kayıpta $r=y-\sigma(F)$.

Regresyon stump'ı için SSE'yi en çok azaltan bölünme, sol/sağ residual toplamları $S_L, S_R$ ile $\frac{S_L^2}{n_L}+\frac{S_R^2}{n_R}$'yi maksimize edendir; M2.4'teki cumsum hilesi burada da çalışır.

## 1 · Elle

<!-- çözüm -->

Boosting'i bir tablo hâlinde izle: her satır bir tur, sütunlar $F$, residual ve stump çıktısı. Stump her turda aynı yerden (2.5) bölüyor ve residual'lar her turda yarıya iniyor ($\eta=0.5$). $\eta=1$ olsaydı tek turda sıfır eğitim hatası olurdu; bunun bir aşırı uyum riski olduğunu düşün.

Bagging kısmında korelasyonun etkisine bak: bağımsız 5 model varyansı 5 kat düşürürken $\rho=0.5$ ile yalnız 1.67 kat düşüyor. $B$'yi sonsuza götürsen de $0.5$'in altına inemezsin.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Boosting (MSE): $x=[1,2,3,4]$, $y=[1,1,3,3]$, $\eta=0.5$, stump eşiği 2.5**

- `F0`: $F_0$ (sabit başlangıç)
- `r0`: 1. örneğin residual'ı $y_1-F_0$
- `F1`: $F_1(x_1)=F_0+\eta\,h_1(x_1)$
- `F2`: $F_2(x_1)$
- `mse2`: 2 tur sonra eğitim MSE'si

**Bagging**

- `vote`: 3 sınıflandırıcının oyları $[1,0,1]$: çoğunluk
- `avgp`: Olasılıkları $[0.9,0.4,0.6]$: ortalama
- `vind`: $B=5$ bağımsız model, her biri $\sigma^2=1$: ortalamanın varyansı
- `vcor`: Aynısı, korelasyon $\rho=0.5$

In [ ]:
cevaplar = dict(
    F0=None,
    r0=None,
    F1=None,
    F2=None,
    mse2=None,
    vote=None,
    avgp=None,
    vind=None,
    vcor=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"F0":[2,0],"r0":[-1,0],"F1":[1.5,0.001],"F2":[1.25,0.001],"mse2":[0.0625,0.0005],"vote":[1,0],"avgp":[0.633333,0.002],"vind":[0.2,0.001],"vcor":[0.6,0.001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, tek özellik, MSE ile gradient boosting.

- `fit_stump(xs, rs)` → `(t, sol_değer, sağ_değer)`; SSE'yi en çok azaltan orta-nokta eşiği, yapraklar residual ortalaması. Eşitlikte küçük eşik. Hiç bölünme yoksa `(None, ort, ort)`.
- `gbm_fit(xs, ys, rounds, lr)` → model (yapısı serbest).
- `gbm_predict(model, x)` → tek bir tahmin.

In [ ]:
%%writefile stage_b.py
def fit_stump(xs, rs):
    ...


def gbm_fit(xs, ys, rounds, lr):
    ...


def gbm_predict(model, x):
    ...


m = gbm_fit([1, 2, 3, 4], [1, 1, 3, 3], 2, 0.5)
print([gbm_predict(m, x) for x in [1, 2, 3, 4]])   # [1.25, 1.25, 2.75, 2.75]

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

Çok özellikli, NumPy.

- `best_reg_stump(X, r)` → `(j, t, sol, sağ)`; tüm özellik ve eşikler **döngüsüz**. Sıralı residual'ların cumsum'u $S_L$, sayaç $n_L$; kazanç $\frac{S_L^2}{n_L}+\frac{S_R^2}{n_R}$. Eşit değer pozisyonları geçersiz. Eşitlikte küçük özellik, sonra küçük eşik. Bölünme yoksa `None`.
- `gbm_np(X, y, rounds, lr)` → model; `gbm_predict_np(model, X)` → `(n,)`. Her ikisinde de yalnız **tur** döngüsü.

Kontrol: $y=\sin(2x_0)+x_1^2$ gibi doğrusal olmayan bir fonksiyonda 200 turda test $R^2\ge0.9$.

In [ ]:
%%writefile stage_c.py
import numpy as np


def best_reg_stump(X, r):
    """→ (j, t, sol, sağ) ya da None. Döngü yok."""
    ...


def gbm_np(X, y, rounds, lr):
    ...


def gbm_predict_np(model, X):
    ...


rng = np.random.default_rng(0)
X = rng.uniform(-2, 2, size=(400, 2))
y = np.sin(2 * X[:, 0]) + X[:, 1] ** 2
m = gbm_np(X, y, 100, 0.2)
print(np.mean((gbm_predict_np(m, X) - y) ** 2))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Boosting'in tek kayba özgü kısmı pseudo-residual. Onu elle türetmek yerine autograd'a bırak:

```python
F = F.detach().requires_grad_(True)
g, = torch.autograd.grad(loss_fn(F, y).sum(), F)
r = -g
```

Başlangıçta verilen `fit_reg_stump` (torch, döngüsüz) ağaç oturtmayı yapıyor; sen yalnız boosting döngüsünü kuruyorsun.

- `pseudo_residuals(F, y, loss_fn)` → $-\partial\sum L/\partial F$.
- `gbm_classify(X, y, rounds=200, lr=0.3)` → model; logistic kayıp (`F.binary_cross_entropy_with_logits`, `reduction="none"`), $F_0=\log\frac{\bar y}{1-\bar y}$.
- `gbm_logits(model, X)` → `(n,)`.

Kontrol: MSE kaybında residual'ın $-2(F-y)$, logistic'te $y-\sigma(F)$ olduğunu ve iç içe çemberleri ≥ 0.9 doğrulukla ayırdığını doğrular (stump'ların toplamı $x_0^2+x_1^2$ gibi toplamsal sınırları temsil edebilir).

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F


def fit_reg_stump(X, r):
    """(Hazır) X: (n, d), r: (n,) → (j, t, sol, sağ). SSE'yi en çok azaltan stump."""
    n, d = X.shape
    Xs, order = torch.sort(X, dim=0, stable=True)
    Rs = r[order]
    SL = Rs.cumsum(0)[:-1]
    SR = Rs.sum(0) - SL
    nl = torch.arange(1, n, dtype=X.dtype)[:, None]
    gain = torch.where(Xs[:-1] == Xs[1:], -torch.inf, SL ** 2 / nl + SR ** 2 / (n - nl))
    k = int(gain.T.reshape(-1).argmax()); j, i = divmod(k, n - 1)
    return j, float((Xs[i, j] + Xs[i + 1, j]) / 2), float(SL[i, j] / (i + 1)), float(SR[i, j] / (n - i - 1))


def stump(st, X):
    j, t, l, r = st
    return torch.where(X[:, j] <= t, l, r).to(X.dtype)


def pseudo_residuals(F, y, loss_fn):
    """−∂ Σ loss_fn(F, y) / ∂F, autograd ile."""
    ...


def gbm_classify(X, y, rounds=200, lr=0.3):
    ...


def gbm_logits(model, X):
    ...

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Bagging: $B$ bootstrap örnekleminin her birine bir regresyon stump'ı. Döngü yerine her örneklemi bir **ağırlık vektörü** olarak gör: $w_{b,i}$ = örnek $i$'nin $b$. örneklemde kaç kez çekildiği. Ağırlık matrisi `(B, n)`: `torch.zeros(B, n).scatter_add_(1, idx, ones)`.

Ağırlıklı kazanç $\frac{S_L^2}{W_L}+\frac{S_R^2}{W_R}$; $S$ ağırlıklı residual toplamı, $W$ ağırlık toplamı. Sıralama tüm $b$'ler için aynı (X değişmiyor), cumsum `(B, n-1, d)` tensöründe.

- `bagged_stumps(X, y, B, generator)` → `(j, t, sol, sağ)`, her biri `(B,)`. Bir tarafın ağırlığı 0 olan bölünmeler geçersiz. **Döngü yok.**
- `bagged_predict(model, X)` → `(n,)` ortalama tahmin. **Döngü yok.**

In [ ]:
%%writefile stage_e.py
import torch


def bagged_stumps(X, y, B, generator):
    """→ (j, t, sol, sağ), her biri (B,). Döngü yok."""
    ...


def bagged_predict(model, X):
    """→ (n,) B stump'ın ortalaması. Döngü yok."""
    ...


g = torch.Generator().manual_seed(0)
X = torch.rand(200, 3, generator=g)
y = torch.sin(6 * X[:, 0]) + 0.1 * torch.randn(200, generator=g)
model = bagged_stumps(X, y, 100, g)
print(((bagged_predict(model, X) - y) ** 2).mean())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Breiman (2001): rastgele ormanlar

[Breiman — Random Forests (Machine Learning 2001)](https://link.springer.com/article/10.1023/A:1010933404324)

Rastgele ormanı M2.4'teki ağacınla kur (bootstrap + düğüm başına m rastgele öznitelik). Makalenin kuramsal ilişkisini ölç: genelleme hatası üst sınırı ρ̄(1 − s²)/s² (ağaçlar arası ortalama korelasyon ρ̄ ve güç s); m'yi değiştirerek ρ̄ ve s'yi out-of-bag tahminlerle hesapla (Bölüm 2.3). OOB hatasının test hatasını izlediğini ve permütasyon önemini (Bölüm 10) göster.


**Adımlar**

1. Korelasyonu ağaçların marj fonksiyonları üzerinden hesapla.

**Kabul ölçütleri**

- m arttıkça ρ̄ ve s birlikte artar; en iyi m ödünleşimde.
- OOB hatası test hatasına yakın.

## Makale ödevi: Friedman (2001): gradyan artırma makinesi

[Friedman — Greedy Function Approximation: A Gradient Boosting Machine (Annals of Statistics 2001)](https://projecteuclid.org/journals/annals-of-statistics/volume-29/issue-5/Greedy-function-approximation-A-gradient-boosting-machine/10.1214/aos/1013203451.full)

Gradyan artırmayı (Algoritma 1) kayıp fonksiyonundan bağımsız yaz; kare, mutlak (LAD) ve Huber kayıpları için sözde kalıntıları ve yaprak değerlerini (Bölüm 4) uygula. Aykırı değerli bir regresyon probleminde üç kaybı karşılaştır; küçültme (ν, Bölüm 5) ve alt örneklemenin (stokastik artırma) test hatasına etkisini çiz.


**Adımlar**

1. Ağaç derinliği J ∈ {2, 4, 8} ile etkileşim düzeyini değiştir.

**Kabul ölçütleri**

- Huber/LAD aykırı değerlerde kareden iyi.
- Küçük ν daha çok ağaçla daha iyi test hatası.

## Kaynaklar

- [ISLP — §8.2 Bagging, Random Forests, Boosting](https://www.statlearning.com/) — doğrulanmadı
- [Elements of Statistical Learning — Bölüm 10 (Boosting) ve 15 (Random Forests)](https://hastie.su.domains/ElemStatLearn/) — doğrulanmadı
- [Friedman — Greedy Function Approximation: A Gradient Boosting Machine](https://jerryfriedman.su.domains/ftp/trebst.pdf) — ©
- [Chen & Guestrin — XGBoost: A Scalable Tree Boosting System](https://arxiv.org/abs/1603.02754) — arXiv